# Assignment 06 · Notebook 01: giá cổ phiếu AMZN

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Dữ liệu ngày của Amazon.com (NASDAQ: AMZN) từ phiên đầu tiên sau IPO (15/05/1997) đến 24/09/2024, tải từ
Yahoo Finance bằng `yfinance` và lưu sẵn ở `data/amzn/AMZN.csv` để tái lập. Notebook phân tích dữ liệu, kiểm định
tính dừng, rồi dựng cửa sổ trượt và chia theo thời gian.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from scipy import stats
from a06.data_stock import CSV, FEATURES, WINDOW, acf_pacf, adf, load_amzn, prepare_stock
from a06.export import save_metrics, write_dat, write_matrix_dat

raw = pd.read_csv(CSV, parse_dates=["Date"], index_col="Date")
df = load_amzn()
print("Số phiên gốc:", len(raw), raw.index[0].date(), "→", raw.index[-1].date())
print("Sau khi thêm đặc trưng (bỏ", len(raw) - len(df), "phiên đầu thiếu MA50):", len(df))
raw.tail()

Số phiên gốc: 6885 1997-05-15 → 2024-09-24
Sau khi thêm đặc trưng (bỏ 49 phiên đầu thiếu MA50): 6836


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2024-09-18,186.449997,188.800003,185.059998,186.429993,186.429993,34448100
2024-09-19,190.039993,190.990005,188.470001,189.869995,189.869995,39543200
2024-09-20,190.229996,191.839996,187.410004,191.600006,191.600006,100378600
2024-09-23,191.639999,194.449997,190.570007,193.880005,193.880005,36993100
2024-09-24,194.270004,195.369995,190.130005,193.960007,193.960007,43478900


## Thống kê mô tả

Return là log-return ngày $r_t = \ln(P_t / P_{t-1})$; Volatility20 là độ lệch chuẩn return 20 phiên, quy ra năm
(nhân $\sqrt{252}$); RSI14 theo Wilder.

In [2]:
cols = ["Close", "Open", "High", "Low", "Volume", "Return", "MA20", "MA50", "Volatility20", "RSI14"]
desc = df[cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
desc.columns = ["mean", "std", "min", "q25", "median", "q75", "max"]
desc["kurt"] = df[cols].kurt()          # độ nhọn dư (Fisher): phân phối chuẩn bằng 0
desc

,mean,std,min,q25,median,q75,max,kurt
Close,3.841724e+01,5.470278e+01,1.020830e-01,2.118500e+00,8.909750e+00,5.829500e+01,2.000000e+02,0.597188
Open,3.842571e+01,5.472701e+01,1.026040e-01,2.112750e+00,8.918250e+00,5.843850e+01,2.000900e+02,0.600430
High,3.887849e+01,5.534711e+01,1.026040e-01,2.155937e+00,9.011000e+00,5.879412e+01,2.012000e+02,0.591271
Low,3.793386e+01,5.404387e+01,9.687500e-02,2.075000e+00,8.805750e+00,5.800337e+01,1.990500e+02,0.606596
Volume,1.375936e+08,1.373805e+08,9.744000e+06,6.478650e+07,1.016970e+08,1.562825e+08,2.086584e+09,34.240347
Return,1.089502e-03,3.464019e-02,-2.845678e-01,-1.300386e-02,4.865104e-04,1.455276e-02,2.961811e-01,9.706378
MA20,3.815894e+01,5.438970e+01,1.047657e-01,2.119406e+00,8.956425e+00,5.744897e+01,1.935985e+02,0.590391
MA50,3.777303e+01,5.396307e+01,8.864066e-02,2.121905e+00,8.832495e+00,5.346288e+01,1.878794e+02,0.602860
Volatility20,4.615328e-01,3.037010e-01,7.762184e-02,2.431955e-01,3.702916e-01,5.669596e-01,1.592595e+00,1.520779
RSI14,5.351358e+01,1.205837e+01,1.775286e+01,4.477010e+01,5.380618e+01,6.202116e+01,9.421413e+01,-0.407076


## Biến động theo các cuộc khủng hoảng

Biến động 20 phiên (năm hoá) lớn nhất trong ba giai đoạn: bong bóng dot-com, khủng hoảng tài chính 2008 và cú sốc
COVID-19, cùng tương quan giữa khối lượng, biến động và mức giá.

In [3]:
crises = {"dotcom": ("2000-01-01", "2001-12-31"), "gfc": ("2008-01-01", "2009-06-30"), "covid": ("2020-02-01", "2020-06-30")}
crisis = {}
for k, (a, b) in crises.items():
    s = df.loc[a:b, "Volatility20"]
    crisis[k] = {"max": s.max(), "date": str(s.idxmax().date())}
calm = df.loc["2012-01-01":"2019-12-31", "Volatility20"].median()
lev = {"vol_volat": df.Volume.corr(df.Volatility20), "close_volat": df.Close.corr(df.Volatility20)}
print(crisis, "· trung vị 2012-2019:", round(calm, 3), lev)

{'dotcom': {'max': np.float64(1.5868685888937213), 'date': '2001-12-05'}, 'gfc': {'max': np.float64(1.1820990517797032), 'date': '2008-10-15'}, 'covid': {'max': np.float64(0.6796307170102887), 'date': '2020-03-25'}} · trung vị 2012-2019: 0.232 {'vol_volat': np.float64(0.48820500565328234), 'close_volat': np.float64(-0.3440983200597963)}


In [4]:
def year_frac(idx):
    return idx.year + (idx.dayofyear - 1) / 365.25

w = df.resample("W-FRI").last()          # 1 điểm mỗi tuần cho hình toàn giai đoạn
write_dat("amzn_price_weekly", {"x": year_frac(w.index), "close": w.Close, "ma20": w.MA20, "ma50": w.MA50,
                                "vol": w.Volume / 1e6, "volat": w.Volatility20})
d3 = df.iloc[::3]
write_dat("amzn_return", {"x": year_frac(d3.index), "r": 100 * d3.Return})

r = df.Return.to_numpy()
edges = np.linspace(-0.15, 0.15, 61)
hist, _ = np.histogram(r, bins=edges, density=True)
mid = (edges[:-1] + edges[1:]) / 2
write_dat("amzn_return_hist", {"x": 100 * mid, "dens": hist / 100,
                               "gauss": stats.norm.pdf(mid, r.mean(), r.std()) / 100})
corr = df[FEATURES].corr().to_numpy()
write_matrix_dat("amzn_corr", corr)
ret_stats = {"mean": r.mean(), "std": r.std(), "skew": stats.skew(r), "kurt": stats.kurtosis(r),
             "min": r.min(), "max": r.max(), "min_date": str(df.Return.idxmin().date()),
             "max_date": str(df.Return.idxmax().date()), "tail4": float(np.mean(np.abs(r - r.mean()) > 4 * r.std())),
             "tail4_gauss": float(2 * stats.norm.sf(4))}
print({k: v for k, v in ret_stats.items()})
pd.DataFrame(corr, index=FEATURES, columns=FEATURES).round(2)

{'mean': np.float64(0.0010895017005785749), 'std': np.float64(0.03463765918924615), 'skew': np.float64(0.43986608910163366), 'kurt': np.float64(9.698402017009444), 'min': np.float64(-0.2845677753224223), 'max': np.float64(0.29618109550941135), 'min_date': '2001-07-24', 'max_date': '2001-11-26', 'tail4': 0.00994733762434172, 'tail4_gauss': 6.334248366623973e-05}


,Close,Open,High,Low,Volume,Return,MA20,RSI14
Close,1.00,1.00,1.00,1.00,-0.31,-0.00,1.00,0.03
Open,1.00,1.00,1.00,1.00,-0.31,-0.01,1.00,0.03
High,1.00,1.00,1.00,1.00,-0.30,-0.01,1.00,0.03
Low,1.00,1.00,1.00,1.00,-0.31,-0.01,1.00,0.03
Volume,-0.31,-0.31,-0.30,-0.31,1.00,0.13,-0.30,0.05
Return,-0.00,-0.01,-0.01,-0.01,0.13,1.00,-0.01,0.31
MA20,1.00,1.00,1.00,1.00,-0.30,-0.01,1.00,0.00
RSI14,0.03,0.03,0.03,0.03,0.05,0.31,0.00,1.00


## Tính dừng: kiểm định ADF và ACF/PACF

Giá có xu hướng nên không dừng; return thì dừng. Đó là lý do mô hình dự báo return chứ không dự báo mức giá.

In [5]:
adf_res = {"close": adf(df.Close), "return": adf(df.Return)}
for k, v in adf_res.items():
    print(f"{k:6s} ADF = {v['stat']:8.3f}  p = {v['p']:.4g}  (ngưỡng 5 %: {v['crit5']:.3f})")
acf_c, pacf_c, band = acf_pacf(df.Close)
acf_r, pacf_r, _ = acf_pacf(df.Return)
lags = np.arange(len(acf_c))
write_dat("amzn_acf", {"lag": lags[1:], "acf_c": acf_c[1:], "pacf_c": pacf_c[1:], "acf_r": acf_r[1:], "pacf_r": pacf_r[1:]})
print("dải tin cậy 95 %: ±", round(band, 4), "· ACF giá ở trễ 40:", round(acf_c[40], 3),
      "· số trễ return vượt dải:", int(np.sum(np.abs(acf_r[1:]) > band)))

D:\Python\Intelligent-System-Development\src\Assignment 06\notebooks\..\a06\data_stock.py:57: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, nobs, crit, _ = adfuller(series.dropna(), autolag="AIC")


close  ADF =    1.246  p = 0.9963  (ngưỡng 5 %: -2.862)
return ADF =  -13.987  p = 4.077e-26  (ngưỡng 5 %: -2.862)
dải tin cậy 95 %: ± 0.0237 · ACF giá ở trễ 40: 0.97 · số trễ return vượt dải: 11


D:\Python\Intelligent-System-Development\src\Assignment 06\notebooks\..\a06\data_stock.py:57: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, p, lags, nobs, crit, _ = adfuller(series.dropna(), autolag="AIC")


## Cửa sổ trượt và chia theo thời gian

Cửa sổ 30 phiên × 8 đặc trưng, nhãn là return phiên kế tiếp. Chia 70/15/15 theo thời gian, không xáo trộn.
Mọi scaler fit **chỉ trên train**; giá trong cửa sổ chia cho Close ngày cuối cửa sổ nên mô hình không phải
ngoại suy ra ngoài khoảng giá đã thấy.

In [6]:
S = prepare_stock(df)
split = {}
for k in ("train", "val", "test"):
    d = S["dates"][k]
    split[k] = {"n": len(d), "start": str(d[0].date()), "end": str(d[-1].date()),
                "p_min": S["p_next"][k].min(), "p_max": S["p_next"][k].max()}
    print(k, split[k])
assert S["dates"]["train"][-1] < S["dates"]["val"][0] < S["dates"]["test"][0], "rò rỉ thời gian"
Xtr = S["X"]["train"].reshape(-1, len(FEATURES))
print("train sau chuẩn hoá: mean ≈", Xtr.mean(0).round(3), "\nstd ≈", Xtr.std(0).round(3))
print("return test: mean", S["ret"]["test"].mean(), "std", S["ret"]["test"].std(), "· train std", S["ret"]["train"].std())
bounds = {k: float(year_frac(pd.DatetimeIndex([S["dates"][k][0]]))[0]) for k in ("val", "test")}

train {'n': 4764, 'start': '1997-09-08', 'end': '2016-08-11', 'p_min': np.float64(0.15), 'p_max': np.float64(38.562)}
val {'n': 1020, 'start': '2016-08-12', 'end': '2020-08-31', 'p_min': np.float64(35.953499), 'p_max': np.float64(172.548004)}
test {'n': 1022, 'start': '2020-09-01', 'end': '2024-09-24', 'p_min': np.float64(81.82), 'p_max': np.float64(200.0)}
train sau chuẩn hoá: mean ≈ [ 0. -0. -0. -0.  0.  0.  0. -0.] 
std ≈ [1. 1. 1. 1. 1. 1. 1. 1.]
return test: mean 0.00011445842124413453 std 0.02251299525301967 · train std 0.03913621152953041


In [7]:
save_metrics("amzn", {
    "n_raw": len(raw), "n_feat": len(df), "start": str(raw.index[0].date()), "end": str(raw.index[-1].date()),
    "window": WINDOW, "n_features": len(FEATURES), "n_windows": sum(v["n"] for v in split.values()),
    "desc": {c: desc.loc[c].to_dict() for c in cols}, "ret": ret_stats, "adf": adf_res,
    "crisis": crisis, "calm_volat": calm, "lev": lev,
    "acf_band": band, "acf_c40": acf_c[40], "acf_r_out": int(np.sum(np.abs(acf_r[1:]) > band)),
    "split": split, "bounds": bounds,
    "ret_std": {k: float(S["ret"][k].std()) for k in ("train", "val", "test")},
    "p_max_ratio": split["test"]["p_max"] / split["train"]["p_max"]})
print("đã ghi outputs/metrics/amzn.json")

đã ghi outputs/metrics/amzn.json
